In [ ]:
!gdown 1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I

Downloading...
From: https://drive.google.com/uc?id=1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I
To: /content/df_train.csv
100% 286k/286k [00:00<00:00, 9.55MB/s]


In [ ]:
!gdown 1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG

Downloading...
From: https://drive.google.com/uc?id=1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG
To: /content/df_test.csv
100% 71.9k/71.9k [00:00<00:00, 13.3MB/s]


In [ ]:
!nvidia-smi

Fri Jun 12 08:15:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pandas as pd

df = pd.read_csv("df_train.csv")
df

,dwt_mean,dwt_max,dwt_min,dwt_median,dwt_q1,dwt_q3,dwt_iqr,mfcc_mean,mfcc_std,mfcc_max,...,rms_min,rms_median,rms_var,rms_skew,rms_kurtosis,rms_q1,rms_q3,rms_range,shannon_entropy,audio_type
0,1.206750e-08,0.013878,-0.014668,-1.500644e-11,-0.000028,0.000026,0.000055,-81.473040,295.34805,152.37076,...,9.717603e-08,0.000048,1.730892e-07,1.283449,0.308629,1.093666e-05,0.000524,0.001483,8.866710,myocardial
1,5.968118e-07,0.039521,-0.040875,1.670743e-11,-0.000016,0.000017,0.000033,-82.751785,298.57016,158.75189,...,1.647872e-07,0.000034,7.909553e-08,4.731361,22.492230,9.232333e-06,0.000076,0.001755,5.972518,myocardial
2,3.870791e-07,0.032473,-0.023627,7.594341e-12,-0.000017,0.000016,0.000033,-81.980400,297.24650,171.03497,...,1.124997e-07,0.000082,3.880997e-08,3.890662,20.913025,3.342536e-06,0.000215,0.001343,7.658798,myocardial
3,-4.941957e-07,0.131609,-0.118817,-2.006760e-11,-0.000014,0.000014,0.000028,-77.834270,280.35214,168.92433,...,9.554750e-08,0.000026,5.670202e-06,4.185934,17.443203,5.695775e-06,0.000075,0.015121,7.720282,myocardial
4,4.204849e-07,0.002665,-0.002124,-7.600626e-11,-0.000031,0.000029,0.000060,-82.015420,298.52830,97.61485,...,8.227275e-07,0.000036,3.165749e-09,0.953618,-0.323343,1.287166e-05,0.000091,0.000202,9.517833,myocardial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,-1.585129e-08,0.011102,-0.009699,-2.031497e-09,-0.000019,0.000020,0.000039,-82.583570,297.38315,139.96802,...,1.878590e-06,0.000031,8.288504e-09,3.186870,11.402454,1.837468e-05,0.000060,0.000525,7.771392,normal
444,5.693161e-07,0.022825,-0.028649,-4.278836e-11,-0.000013,0.000013,0.000027,-83.153275,299.87430,200.29291,...,1.362070e-07,0.000027,1.346874e-07,3.889586,15.481110,1.185905e-05,0.000067,0.002202,5.310026,normal
445,4.123294e-09,0.060122,-0.060773,-4.331842e-10,-0.000004,0.000004,0.000008,-84.163050,296.94974,180.51411,...,7.595175e-07,0.000011,9.946041e-07,2.224082,3.260276,5.449809e-06,0.000059,0.003769,5.532661,normal
446,-3.339084e-08,0.089756,-0.102157,1.016094e-11,-0.000002,0.000002,0.000004,-82.164860,292.40936,203.16748,...,1.403314e-07,0.000011,1.463136e-06,5.434349,30.395962,8.566282e-07,0.000142,0.008902,6.255238,normal


In [ ]:
import numpy as np
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, Activation, MaxPooling1D, LSTM, Dense, Dropout, Flatten

In [ ]:
model = Sequential()

input_shape = (51, 1)
num_classes = 2

model.add(Conv1D(filters=16, kernel_size=3, padding='same', input_shape=input_shape))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

model.add(Conv1D(filters=16, kernel_size=3, padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))

model.add(LSTM(32, return_sequences=True))

model.add(LSTM(128, return_sequences=False))

model.add(Dense(64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))

model.add(Dense(num_classes, activation='softmax'))

opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 51, 16)         │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 51, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 51, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 25, 16)         │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 25, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 25, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 12, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 12, 32)         │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 98,322 (384.07 KB)

 Trainable params: 98,130 (383.32 KB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
X = df.drop('audio_type', axis=1).values
y = df['audio_type'].values

# Encode categorical labels to numerical
from sklearn.preprocessing import LabelEncoder, StandardScaler
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y)

# One-hot encode numerical labels
y_one_hot = tf.keras.utils.to_categorical(y_numerical, num_classes=2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for NaN values in features
if np.isnan(X_scaled).any():
    print("Warning: NaN values found in scaled features (X_scaled). This can cause 'nan' loss during training.")
else:
    print("No NaN values found in scaled features (X_scaled).")

No NaN values found in scaled features (X_scaled).


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
import numpy as np

X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

fold_no = 1
fold_metrics = {
    'accuracy': [],
    'sensitivity': [],
    'specificity': [],
    'auroc': []
}

print("Starting 10-Fold Cross-Validation on Training Set...")

for train_index, val_index in kfold.split(X_train_full):
    X_fold_train, X_fold_val = X_train_full[train_index], X_train_full[val_index]
    y_fold_train, y_fold_val = y_train_full[train_index], y_train_full[val_index]

    print(f'------------------------------------------------------------------------')
    print(f'Training for Fold {fold_no} ...')

    # Re-initialize or fit model
    model.fit(
        X_fold_train,
        y_fold_train,
        batch_size=32,
        epochs=30,
        verbose=0,
        validation_data=(X_fold_val, y_fold_val)
    )

    # Generate predictions for the validation fold
    y_pred_probs = model.predict(X_fold_val, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_fold_val, axis=1)

    # Calculate Metrics
    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Sensitivity & Specificity
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    auroc = roc_auc_score(y_fold_val[:, 1], y_pred_probs[:, 1])

    # Store results
    fold_metrics['accuracy'].append(acc)
    fold_metrics['sensitivity'].append(sens)
    fold_metrics['specificity'].append(spec)
    fold_metrics['auroc'].append(auroc)

    print(f'Fold {fold_no} Results: Acc={acc:.4f}, Sens={sens:.4f}, Spec={spec:.4f}, AUROC={auroc:.4f}')
    fold_no += 1

# Summary
print(f'\n------------------------------------------------------------------------')
print(f'Summary Results (Mean ± STD) over 10 Folds:')
for metric, values in fold_metrics.items():
    print(f'{metric.capitalize()}: {np.mean(values)*100:.2f}% ± {np.std(values)*100:.2f}%')

Starting 10-Fold Cross-Validation on Training Set...
------------------------------------------------------------------------
Training for Fold 1 ...
Fold 1 Results: Acc=0.6341, Sens=0.2778, Spec=0.9130, AUROC=0.7174
------------------------------------------------------------------------
Training for Fold 2 ...
Fold 2 Results: Acc=0.7805, Sens=0.5625, Spec=0.9200, AUROC=0.9400
------------------------------------------------------------------------
Training for Fold 3 ...
Fold 3 Results: Acc=0.7317, Sens=1.0000, Spec=0.4762, AUROC=0.9476
------------------------------------------------------------------------
Training for Fold 4 ...
Fold 4 Results: Acc=0.9000, Sens=0.9565, Spec=0.8235, AUROC=0.9693
------------------------------------------------------------------------
Training for Fold 5 ...
Fold 5 Results: Acc=0.9250, Sens=0.9524, Spec=0.8947, AUROC=0.9900
------------------------------------------------------------------------
Training for Fold 6 ...
Fold 6 Results: Acc=0.9250, Se

In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 75.56%
Sensitivity: 68.18%
Specificity: 82.61%
AUROC: 0.8182


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 77.78%
Sensitivity: 86.36%
Specificity: 69.57%
AUROC: 0.8498


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 77.78%
Sensitivity: 90.91%
Specificity: 65.22%
AUROC: 0.8379


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 71.11%
Sensitivity: 50.00%
Specificity: 91.30%
AUROC: 0.8221


In [ ]:
# 3. Final Evaluation on Test Data
# Train final model on full training set and evaluate on the 30% held-out test set
print("\nTraining Final Model on full Training Set...")
final_model = model
final_model.fit(X_train_full, y_train_full, batch_size=32, epochs=30, verbose=0)

loss, accuracy = final_model.evaluate(X_test, y_test, verbose=0)

y_pred_probs = final_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

# Calculate AUROC
auroc = roc_auc_score(y_test[:, 1], y_pred_probs[:, 1])

print(f"\n--- Final MIDs Results (Test Set) ---")
print(f"Accuracy: {accuracy*100:.2f}%")
print(f"Sensitivity: {sensitivity*100:.2f}%")
print(f"Specificity: {specificity*100:.2f}%")
print(f"AUROC: {auroc:.4f}")


Training Final Model on full Training Set...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step

--- Final MIDs Results (Test Set) ---
Accuracy: 73.33%
Sensitivity: 63.64%
Specificity: 82.61%
AUROC: 0.8379


In [ ]:
df_test = pd.read_csv('df_test.csv')

In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test)
specificity_test = tn_test / (tn_test + fp_test)

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

--- Evaluation on df_test --- 
Test Accuracy: 73.21%
Test Sensitivity: 75.00%
Test Specificity: 71.43%
Test AUROC: 0.8117
Confusion Matrix:
[[40 16]
 [14 42]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test)
specificity_test = tn_test / (tn_test + fp_test)

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

--- Evaluation on df_test --- 
Test Accuracy: 75.89%
Test Sensitivity: 73.21%
Test Specificity: 78.57%
Test AUROC: 0.8210
Confusion Matrix:
[[44 12]
 [15 41]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test)
specificity_test = tn_test / (tn_test + fp_test)

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step

--- Evaluation on df_test --- 
Test Accuracy: 75.89%
Test Sensitivity: 75.00%
Test Specificity: 76.79%
Test AUROC: 0.8358
Confusion Matrix:
[[43 13]
 [14 42]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test)
specificity_test = tn_test / (tn_test + fp_test)

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

--- Evaluation on df_test --- 
Test Accuracy: 73.21%
Test Sensitivity: 76.79%
Test Specificity: 69.64%
Test AUROC: 0.8291
Confusion Matrix:
[[39 17]
 [13 43]]


In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

# Evaluate the final model on the test data
loss_test, accuracy_test = final_model.evaluate(X_scaled_test, y_one_hot_test, verbose=0)

# Make predictions on the test data
y_pred_probs_test = final_model.predict(X_scaled_test)
y_pred_test = np.argmax(y_pred_probs_test, axis=1)
y_true_test = np.argmax(y_one_hot_test, axis=1)

# Compute the confusion matrix
conf_matrix_test = confusion_matrix(y_true_test, y_pred_test)
tn_test, fp_test, fn_test, tp_test = conf_matrix_test.ravel()

# Calculate Sensitivity and Specificity
sensitivity_test = tp_test / (tp_test + fn_test)
specificity_test = tn_test / (tn_test + fp_test)

# Calculate AUROC
auroc_test = roc_auc_score(y_one_hot_test[:, 1], y_pred_probs_test[:, 1])

print(f"\n--- Evaluation on df_test --- ")
print(f"Test Accuracy: {accuracy_test*100:.2f}%")
print(f"Test Sensitivity: {sensitivity_test*100:.2f}%")
print(f"Test Specificity: {specificity_test*100:.2f}%")
print(f"Test AUROC: {auroc_test:.4f}")
print(f"Confusion Matrix:\n{conf_matrix_test}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

--- Evaluation on df_test --- 
Test Accuracy: 75.00%
Test Sensitivity: 85.71%
Test Specificity: 64.29%
Test AUROC: 0.8048
Confusion Matrix:
[[36 20]
 [ 8 48]]
